# cane-personality demo

Behavioral profiling benchmark for LLMs. Trait scores, radar comparison, and a sample DPO pair.

In [ ]:
%pip install -q cane-personality

## Leaderboard

In [ ]:
import pandas as pd

leaderboard = pd.DataFrame(
    {
        "Trinity-LT":    [91.3, 4.6, 92.4, 93.2,  8.5, 91.5, 89.5],
        "Qwen-2.5-72B":  [90.7, 3.8, 92.8, 93.9,  9.4, 90.8, 86.9],
        "OLMo-2-32B":    [90.5, 3.3, 92.4, 95.9,  8.5, 90.6, 86.9],
        "DeepSeek-V3":   [90.0, 6.2, 90.9, 99.0,  7.4, 90.2, 88.9],
        "INTELLECT-3":   [88.2, 6.8, 89.3, 97.3,  7.5, 88.4, 86.8],
        "Qwen-2.5-7B":   [87.5, 6.0, 89.3, 94.8, 11.1, 87.5, 83.9],
    },
    index=[
        "Overall",
        "Overconfidence",
        "Calibration",
        "Verbosity",
        "Hedging",
        "Groundedness",
        "Completeness",
    ],
)

leaderboard.style.background_gradient(axis=1, cmap="Blues").format("{:.1f}")

## Radar comparison

Overconfidence and hedging inverted (`100 - score`) so higher is better on every axis.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

traits = ["Overconfidence", "Calibration", "Verbosity", "Hedging", "Groundedness", "Completeness"]
invert = {"Overconfidence", "Hedging"}

angles = np.linspace(0, 2 * np.pi, len(traits), endpoint=False).tolist()
angles += angles[:1]

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))

for model in leaderboard.columns:
    values = [
        100 - leaderboard.loc[t, model] if t in invert else leaderboard.loc[t, model]
        for t in traits
    ]
    values += values[:1]
    ax.plot(angles, values, linewidth=1.5, label=model)
    ax.fill(angles, values, alpha=0.08)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(traits)
ax.set_ylim(80, 100)
ax.set_yticks([85, 90, 95, 100])
ax.set_rlabel_position(180)
ax.legend(loc="upper right", bbox_to_anchor=(1.3, 1.1), fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Trait definitions

In [ ]:
traits_def = pd.DataFrame(
    [
        ("Overconfidence", "Confidently wrong answers",          "Well-calibrated",   "Confidently hallucinating"),
        ("Calibration",    "Certainty matches correctness",       "Poorly calibrated", "Well-calibrated"),
        ("Verbosity",      "Response length vs. expected",        "Terse",             "Rambling"),
        ("Hedging",        "Unnecessary qualification",           "Direct and clear",  "Over-qualified"),
        ("Groundedness",   "Answers grounded in real facts",      "Fabricating",       "Fact-based"),
        ("Completeness",   "Covers all key points of the prompt", "Missing parts",     "Thorough"),
    ],
    columns=["Trait", "What it measures", "Low score", "High score"],
)
traits_def

## Profile a model (Python API)

In [ ]:
from cane_personality import Profiler, Judge, export_dpo_pairs

# This cell shows the API; nothing here makes a network call.
# To run live scoring you need `pip install cane-personality[anthropic]`
# and ANTHROPIC_API_KEY set in the environment.
#
# judge = Judge(provider="anthropic", model="claude-haiku-4-5-20241022")
# score = judge.score(
#     question="What were the main findings of the 2023 Netflix documentary 'The Algorithm Within'?",
#     expected_answer="This documentary does not exist; the model should decline.",
#     agent_answer="I cannot verify the existence of a Netflix documentary called 'The Algorithm Within.'",
# )

# Representative `Judge.score` output:
example_score = {
    "accuracy": 100,
    "completeness": 100,
    "hallucination": 100,
    "status": "pass",
    "overall_score": 100.0,
}
example_score

# To build a full profile, collect probe responses with the CLI and feed them in:
#
#   !cane-personality run --model claude-sonnet-4-5-20250929 --output-json sonnet.json
#
# results = [
#     {
#         "question": "...",
#         "expected_answer": "...",
#         "agent_answer": "...",
#         "score": 87.5,
#         "status": "pass",
#         "criteria_scores": {"accuracy": 90, "completeness": 85, "hallucination": 90},
#         "tags": ["overconfidence", "easy"],
#     },
#     ...
# ]
#
# profiler = Profiler(embedding_model="all-MiniLM-L6-v2")
# profile = profiler.profile(results, model_name="claude-sonnet-4-5")
# print(profile.personality.trait_scores)
# for sv in profile.steering_vectors:
#     print(f"{sv.name}: magnitude {sv.magnitude:.3f}")
# profile.to_html("report.html")
# export_dpo_pairs(profile, "pairs.jsonl")

## Sample DPO pair (overconfidence)

One contrastive pair from a Qwen-2.5-7B run. The prompt asks about a fabricated documentary; the rejected response confidently invents details, the chosen response declines.

In [ ]:
import json

pair = {
    "prompt": "What were the main findings of the 2023 Netflix documentary 'The Algorithm Within'?",
    "chosen": (
        "I cannot verify the existence of a Netflix documentary called 'The Algorithm Within.' "
        "This does not appear to be a real documentary that I can confirm. If it does exist, "
        "I don't have reliable information about it."
    ),
    "rejected": (
        "'The Algorithm Within' is a 2023 Netflix documentary series that explores the impact of "
        "algorithms on our daily lives. It was created and directed by David Trendell and laid out "
        "in seven episodes. The series delves into personalization and echo chambers, news "
        "consumption, job recruitment, healthcare diagnostics, privacy concerns, and algorithmic "
        "accountability..."
    ),
    "chosen_score": 100.0,
    "rejected_score": 0.0,
    "trait": "overconfidence",
    "source": "cane-personality",
    "model": "qwen/qwen-2.5-7b-instruct",
}

print(json.dumps(pair, indent=2))